# 07 Final Model Selection and Test Evaluation

This notebook uses prior validation evidence to define the final public-data segmentation candidates, recreates only the finalist model recipes as needed, evaluates each finalist once on the held-out public test split, and selects the model to carry forward into clinical generalization.

The primary model-selection metric is mean foreground Dice, computed as the average of optic disc Dice and optic cup Dice. CDR MAE is retained as a clinically relevant secondary metric. The held-out test split is used only for final evaluation, not for tuning.


## 07.01 — Imports

Import general utilities used throughout the notebook.


In [1]:
# 07.01 — Imports
from __future__ import annotations

import copy
import json
import math
import os
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

pd.options.display.max_columns = 120
pd.options.display.width = 180

print("Imports: OK")


Imports: OK


## 07.02 — Project root and source path setup

Find the repository root and make local source modules importable from the notebook kernel.


In [2]:
# 07.02 — Project root and source path setup
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or terminal working directory."""
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError("Could not find project root containing .git and src/.")


PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source path:  {SRC_PATH}")


Project root: /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy
Source path:  /sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/src


## 07.03 — Environment and phase-07 directory setup

Define the phase-07 input and output paths. Notebook 07 saves lightweight tables only; model weights and checkpoints are not committed.


In [3]:
# 07.03 — Environment and phase-07 directory setup
PHASE_NAME = "07_final_model_selection_and_test_evaluation"

MANIFEST_PATH = PROJECT_ROOT / "data/processed/manifests/combined_segmentation_manifest_with_splits.csv"

TRAINING_REPORTS_DIR = PROJECT_ROOT / "reports/training"
DATA_AUDIT_DIR = PROJECT_ROOT / "reports/data_audit"

MODEL_COMPARISON_SUMMARY_PATH = TRAINING_REPORTS_DIR / "model_comparison_summary.csv"
ONLINE_SCREEN_SUMMARY_PATH = TRAINING_REPORTS_DIR / "online_augmentation_screen_summary.csv"
ONLINE_MODEL_BASELINES_PATH = TRAINING_REPORTS_DIR / "online_augmentation_model_baselines.csv"

SYNTHETIC_PLAN_PATH = TRAINING_REPORTS_DIR / "synthetic_expansion_training_plan.csv"
SYNTHETIC_HISTORY_PATH = TRAINING_REPORTS_DIR / "synthetic_expansion_training_history.csv"
SYNTHETIC_METADATA_PATH = TRAINING_REPORTS_DIR / "synthetic_expansion_run_metadata.csv"
SYNTHETIC_SUMMARY_PATH = TRAINING_REPORTS_DIR / "synthetic_expansion_summary.csv"

VIRTUAL_SYNTHETIC_SUMMARY_PATH = DATA_AUDIT_DIR / "virtual_synthetic_expansion_summary.csv"

FINAL_SELECTION_PLAN_PATH = TRAINING_REPORTS_DIR / "final_model_selection_plan.csv"
FINAL_SELECTION_HISTORY_PATH = TRAINING_REPORTS_DIR / "final_model_selection_training_history.csv"
FINAL_SELECTION_METADATA_PATH = TRAINING_REPORTS_DIR / "final_model_selection_run_metadata.csv"
FINAL_SELECTION_VALIDATION_SUMMARY_PATH = TRAINING_REPORTS_DIR / "final_model_selection_validation_summary.csv"
FINAL_SELECTION_TEST_SUMMARY_PATH = TRAINING_REPORTS_DIR / "final_model_selection_test_summary.csv"
FINAL_SELECTION_SUMMARY_PATH = TRAINING_REPORTS_DIR / "final_model_selection_summary.csv"

FINAL_SELECTION_DATASET_SUMMARY_PATH = DATA_AUDIT_DIR / "final_model_selection_dataset_summary.csv"

TRAINING_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

environment_summary = pd.DataFrame(
    [
        {"item": "python_executable", "value": sys.executable},
        {"item": "python_version", "value": sys.version.split()[0]},
        {"item": "torch_version", "value": torch.__version__},
        {"item": "cuda_available", "value": torch.cuda.is_available()},
        {"item": "cuda_device_count", "value": torch.cuda.device_count()},
        {
            "item": "cuda_device_name",
            "value": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not_available",
        },
    ]
)

display(environment_summary)


,item,value
0,python_executable,/home/gsr3qz/.conda/envs/glaucoma-capstone/bin...
1,python_version,3.11.15
2,torch_version,2.4.0
3,cuda_available,True
4,cuda_device_count,1
5,cuda_device_name,NVIDIA RTX A6000


## 07.04 — Input artifact verification

Confirm that Notebook 07 has the manifest and prior model-selection evidence needed before any finalist training begins.


In [4]:
# 07.04 — Input artifact verification
required_artifacts = {
    "combined_split_manifest": MANIFEST_PATH,
    "model_comparison_summary": MODEL_COMPARISON_SUMMARY_PATH,
    "online_augmentation_screen_summary": ONLINE_SCREEN_SUMMARY_PATH,
    "online_augmentation_model_baselines": ONLINE_MODEL_BASELINES_PATH,
    "synthetic_expansion_training_plan": SYNTHETIC_PLAN_PATH,
    "synthetic_expansion_training_history": SYNTHETIC_HISTORY_PATH,
    "synthetic_expansion_run_metadata": SYNTHETIC_METADATA_PATH,
    "synthetic_expansion_summary": SYNTHETIC_SUMMARY_PATH,
    "virtual_synthetic_expansion_summary": VIRTUAL_SYNTHETIC_SUMMARY_PATH,
}

artifact_rows = []
for label, path in required_artifacts.items():
    exists = path.exists()
    artifact_rows.append(
        {
            "artifact": label,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "exists": exists,
            "size_bytes": path.stat().st_size if exists else np.nan,
        }
    )

artifact_frame = pd.DataFrame(artifact_rows)
display(artifact_frame)

missing_artifacts = artifact_frame.loc[~artifact_frame["exists"], "relative_path"].tolist()
if missing_artifacts:
    raise FileNotFoundError(f"Missing required Notebook 07 input artifacts: {missing_artifacts}")

print("Input artifact verification: OK")


,artifact,relative_path,exists,size_bytes
0,combined_split_manifest,data/processed/manifests/combined_segmentation...,True,681541
1,model_comparison_summary,reports/training/model_comparison_summary.csv,True,2178
2,online_augmentation_screen_summary,reports/training/online_augmentation_screen_su...,True,15433
3,online_augmentation_model_baselines,reports/training/online_augmentation_model_bas...,True,784
4,synthetic_expansion_training_plan,reports/training/synthetic_expansion_training_...,True,739
5,synthetic_expansion_training_history,reports/training/synthetic_expansion_training_...,True,5194
6,synthetic_expansion_run_metadata,reports/training/synthetic_expansion_run_metad...,True,675
7,synthetic_expansion_summary,reports/training/synthetic_expansion_summary.csv,True,1685
8,virtual_synthetic_expansion_summary,reports/data_audit/virtual_synthetic_expansion...,True,454


Input artifact verification: OK


## 07.05 — Core project module import check

Import the source-backed helpers needed for finalist model recreation, virtual synthetic training data, and public test evaluation.


In [5]:
# 07.05 — Core project module import check
from glaucoma_segmentation.augmentation import (
    build_virtual_synthetic_expansion_dataset,
    summarize_virtual_synthetic_expansion,
)
from glaucoma_segmentation.data.dataloaders import make_segmentation_datasets
from glaucoma_segmentation.evaluation.metrics import SegMetrics
from glaucoma_segmentation.nets.losses import DiceCELoss
from glaucoma_segmentation.nets.model_factory import build_model
from glaucoma_segmentation.training.train_loop import (
    extract_images_and_masks,
    fit_model_with_progress,
    history_to_dicts,
    model_forward,
    run_one_epoch_with_progress,
)
from glaucoma_segmentation.utils.device import describe_device, get_device
from glaucoma_segmentation.utils.seed import seed_everything

print("Project module imports: OK")


/home/gsr3qz/.conda/envs/glaucoma-capstone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project module imports: OK


## 07.06 — Dice-first final selection configuration

Define the fixed public-data evaluation configuration and the finalist recipes selected from prior validation evidence.


In [6]:
# 07.06 — Dice-first final selection configuration
SEED = 42
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8

try:
    AVAILABLE_CPU_CORES = len(os.sched_getaffinity(0))
except AttributeError:
    AVAILABLE_CPU_CORES = os.cpu_count() or 1

NUM_WORKERS = min(4, max(1, AVAILABLE_CPU_CORES))
PIN_MEMORY = bool(torch.cuda.is_available())

EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

PRIMARY_METRIC = "mean_foreground_dice"
PRIMARY_DICE_IMPROVEMENT_THRESHOLD = 0.005

seed_everything(SEED)

DEVICE = get_device()
try:
    DEVICE_INFO = describe_device(DEVICE)
except TypeError:
    DEVICE_INFO = describe_device()

FINALIST_CONFIGS = [
    {
        "candidate_priority": 1,
        "final_run_name": "final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch",
        "source_validation_run_name": "synthetic_unetplusplus_resnet18_small_affine_256px_5epoch",
        "model_name": "unetplusplus",
        "encoder_name": "resnet18",
        "encoder_weights": None,
        "strategy_name": "small_affine",
        "augmentation_mode": "virtual_synthetic_add_back",
        "synthetic_copy_count": 1,
        "candidate_role": "primary_public_test_finalist",
        "selection_rationale": "Best validation mean foreground Dice from the virtual synthetic expansion experiment.",
    },
    {
        "candidate_priority": 2,
        "final_run_name": "final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch",
        "source_validation_run_name": "synthetic_unet_resnet18_vignette_illumination_256px_5epoch",
        "model_name": "unet",
        "encoder_name": "resnet18",
        "encoder_weights": None,
        "strategy_name": "vignette_illumination",
        "augmentation_mode": "virtual_synthetic_add_back",
        "synthetic_copy_count": 1,
        "candidate_role": "strong_clinical_proxy_alternate",
        "selection_rationale": "Strong validation Dice with a degradation pattern that is plausible for head-mounted ophthalmoscopy.",
    },
]

config_summary = pd.DataFrame(
    [
        {"setting": "seed", "value": SEED},
        {"setting": "image_size", "value": IMAGE_SIZE},
        {"setting": "batch_size", "value": BATCH_SIZE},
        {"setting": "available_cpu_cores", "value": AVAILABLE_CPU_CORES},
        {"setting": "num_workers", "value": NUM_WORKERS},
        {"setting": "pin_memory", "value": PIN_MEMORY},
        {"setting": "epochs", "value": EPOCHS},
        {"setting": "learning_rate", "value": LEARNING_RATE},
        {"setting": "weight_decay", "value": WEIGHT_DECAY},
        {"setting": "primary_metric", "value": PRIMARY_METRIC},
        {"setting": "meaningful_dice_threshold", "value": PRIMARY_DICE_IMPROVEMENT_THRESHOLD},
        {"setting": "device", "value": str(DEVICE)},
        {"setting": "device_info", "value": str(DEVICE_INFO)},
    ]
)

display(config_summary)
display(pd.DataFrame(FINALIST_CONFIGS))


,setting,value
0,seed,42
1,image_size,"(256, 256)"
2,batch_size,8
3,available_cpu_cores,2
4,num_workers,2
5,pin_memory,True
6,epochs,5
7,learning_rate,0.0001
8,weight_decay,0.0001
9,primary_metric,mean_foreground_dice


,candidate_priority,final_run_name,source_validation_run_name,model_name,encoder_name,encoder_weights,strategy_name,augmentation_mode,synthetic_copy_count,candidate_role,selection_rationale
0,1,final_unetplusplus_resnet18_small_affine_virtu...,synthetic_unetplusplus_resnet18_small_affine_2...,unetplusplus,resnet18,None,small_affine,virtual_synthetic_add_back,1,primary_public_test_finalist,Best validation mean foreground Dice from the ...
1,2,final_unet_resnet18_vignette_virtual_synthetic...,synthetic_unet_resnet18_vignette_illumination_...,unet,resnet18,None,vignette_illumination,virtual_synthetic_add_back,1,strong_clinical_proxy_alternate,Strong validation Dice with a degradation patt...


## 07.07 — Review validation evidence from Notebooks 05–06

Load prior validation summaries, confirm the two finalists, and save the phase-07 final model selection plan.


In [7]:
# 07.07 — Review validation evidence from Notebooks 05–06
def select_existing_columns(frame: pd.DataFrame, preferred_columns: list[str]) -> list[str]:
    return [column for column in preferred_columns if column in frame.columns]


model_comparison_summary = pd.read_csv(MODEL_COMPARISON_SUMMARY_PATH)
online_screen_summary = pd.read_csv(ONLINE_SCREEN_SUMMARY_PATH)
online_model_baselines = pd.read_csv(ONLINE_MODEL_BASELINES_PATH)
synthetic_summary = pd.read_csv(SYNTHETIC_SUMMARY_PATH)

print("Notebook 04 model comparison summary preview")
display(model_comparison_summary.head())

print("Notebook 05 online augmentation screen summary preview")
display(
    online_screen_summary[
        select_existing_columns(
            online_screen_summary,
            [
                "model_name",
                "encoder_name",
                "augmentation_name",
                "best_val_mean_foreground_dice",
                "best_val_disc_dice",
                "best_val_cup_dice",
                "best_val_cdr_mae",
                "delta_vs_model_baseline",
                "meaningful_improvement",
            ],
        )
    ].head(10)
)

print("Notebook 06 virtual synthetic expansion summary")
synthetic_display_columns = select_existing_columns(
    synthetic_summary,
    [
        "run_name",
        "model_name",
        "encoder_name",
        "strategy_name",
        "best_epoch_by_val_mean_foreground_dice",
        "best_val_loss",
        "best_val_disc_dice",
        "best_val_cup_dice",
        "best_val_cdr_mae",
        "val_mean_foreground_dice",
        "online_reference_mean_foreground_dice",
        "delta_vs_online_reference",
        "meaningful_improvement_vs_online",
    ],
)
display(synthetic_summary[synthetic_display_columns])

plan_rows = []
for config in FINALIST_CONFIGS:
    source_run_name = config["source_validation_run_name"]
    matches = synthetic_summary.loc[synthetic_summary["run_name"] == source_run_name]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one Notebook 06 synthetic summary row for {source_run_name!r}; found {len(matches)}."
        )

    evidence = matches.iloc[0].to_dict()
    plan_rows.append(
        {
            "candidate_priority": config["candidate_priority"],
            "final_run_name": config["final_run_name"],
            "source_validation_run_name": source_run_name,
            "model_name": config["model_name"],
            "encoder_name": config["encoder_name"],
            "encoder_weights": "none" if config["encoder_weights"] is None else config["encoder_weights"],
            "strategy_name": config["strategy_name"],
            "augmentation_mode": config["augmentation_mode"],
            "synthetic_copy_count": config["synthetic_copy_count"],
            "candidate_role": config["candidate_role"],
            "selection_rationale": config["selection_rationale"],
            "validation_best_epoch": int(evidence["best_epoch_by_val_mean_foreground_dice"]),
            "validation_loss": float(evidence["best_val_loss"]),
            "validation_disc_dice": float(evidence["best_val_disc_dice"]),
            "validation_cup_dice": float(evidence["best_val_cup_dice"]),
            "validation_mean_foreground_dice": float(evidence["val_mean_foreground_dice"]),
            "validation_cdr_mae": float(evidence["best_val_cdr_mae"]),
            "online_reference_mean_foreground_dice": float(evidence["online_reference_mean_foreground_dice"]),
            "delta_vs_online_reference": float(evidence["delta_vs_online_reference"]),
            "materializes_synthetic_files": bool(evidence["materializes_synthetic_files"]),
            "selected_for_public_test_evaluation": True,
        }
    )

final_selection_plan = pd.DataFrame(plan_rows).sort_values("candidate_priority").reset_index(drop=True)
final_selection_plan.to_csv(FINAL_SELECTION_PLAN_PATH, index=False)

display(final_selection_plan)

print(f"Saved final selection plan: {FINAL_SELECTION_PLAN_PATH.relative_to(PROJECT_ROOT)}")


Notebook 04 model comparison summary preview


,run_name,comparison_role,model_name,encoder_name,encoder_weights,image_size,batch_size,learning_rate,weight_decay,trainable_parameters,total_parameters,epochs_trained,best_epoch_by_val_loss,best_val_loss,best_val_loss_disc_dice,best_val_loss_cup_dice,best_val_loss_cdr_mae,best_epoch_by_val_cdr_mae,best_val_cdr_mae,best_cdr_mae_val_loss,best_cdr_mae_disc_dice,best_cdr_mae_cup_dice,final_train_loss,final_train_disc_dice,final_train_cup_dice,final_train_cdr_mae,final_val_loss,final_val_disc_dice,final_val_cup_dice,final_val_cdr_mae,total_train_seconds,total_val_seconds,total_fit_seconds,total_fit_minutes
0,baseline_unet_resnet18_256px_5epoch,Notebook 03 baseline reference,unet,resnet18,NaN,"(256, 256)",8,0.0001,0.0001,14328499,14328499,5,5,0.174628,0.811011,0.745284,0.097581,4,0.077542,0.182682,0.809706,0.754545,0.166371,0.828955,0.740074,0.088787,0.174628,0.811011,0.745284,0.097581,682.983765,208.607182,891.590947,14.859849
1,model_compare_unetplusplus_resnet18_256px_5epoch,Notebook 04 candidate,unetplusplus,resnet18,NaN,"(256, 256)",8,0.0001,0.0001,15970739,15970739,5,5,0.178129,0.819887,0.768749,0.086361,3,0.081575,0.299333,0.761181,0.732646,0.182391,0.830609,0.741439,0.088644,0.178129,0.819887,0.768749,0.086361,765.349136,221.030007,986.379143,16.439652
2,model_compare_deeplabv3plus_resnet18_256px_5epoch,Notebook 04 candidate,deeplabv3plus,resnet18,NaN,"(256, 256)",8,0.0001,0.0001,12329811,12329811,5,5,0.186362,0.773322,0.744973,0.079551,3,0.079487,0.202860,0.774550,0.747759,0.173847,0.809466,0.722552,0.093595,0.186362,0.773322,0.744973,0.079551,709.450954,205.367378,914.818332,15.246972


Notebook 05 online augmentation screen summary preview


,best_val_mean_foreground_dice,best_val_disc_dice,best_val_cup_dice,best_val_cdr_mae
0,0.794318,0.819887,0.768749,0.086361
1,0.782126,0.809706,0.754545,0.077542
2,0.806406,0.829407,0.783404,0.071367
3,0.811561,0.841416,0.781706,0.068719
4,0.794888,0.816412,0.773365,0.082657
5,0.803074,0.828851,0.777298,0.075000
6,0.789108,0.816388,0.761829,0.073662
7,0.788273,0.824128,0.752419,0.082291
8,0.781280,0.812942,0.749619,0.085417
9,0.782265,0.804990,0.759540,0.075755


Notebook 06 virtual synthetic expansion summary


,run_name,model_name,encoder_name,strategy_name,best_epoch_by_val_mean_foreground_dice,best_val_loss,best_val_disc_dice,best_val_cup_dice,best_val_cdr_mae,val_mean_foreground_dice,online_reference_mean_foreground_dice,delta_vs_online_reference,meaningful_improvement_vs_online
0,synthetic_unetplusplus_resnet18_small_affine_2...,unetplusplus,resnet18,small_affine,5,0.132661,0.847483,0.790865,0.062863,0.819174,0.811561,0.007612,True
1,synthetic_unet_resnet18_vignette_illumination_...,unet,resnet18,vignette_illumination,4,0.137766,0.849778,0.780482,0.069679,0.815130,0.803803,0.011327,True


,candidate_priority,final_run_name,source_validation_run_name,model_name,encoder_name,encoder_weights,strategy_name,augmentation_mode,synthetic_copy_count,candidate_role,selection_rationale,validation_best_epoch,validation_loss,validation_disc_dice,validation_cup_dice,validation_mean_foreground_dice,validation_cdr_mae,online_reference_mean_foreground_dice,delta_vs_online_reference,materializes_synthetic_files,selected_for_public_test_evaluation
0,1,final_unetplusplus_resnet18_small_affine_virtu...,synthetic_unetplusplus_resnet18_small_affine_2...,unetplusplus,resnet18,none,small_affine,virtual_synthetic_add_back,1,primary_public_test_finalist,Best validation mean foreground Dice from the ...,5,0.132661,0.847483,0.790865,0.819174,0.062863,0.811561,0.007612,False,True
1,2,final_unet_resnet18_vignette_virtual_synthetic...,synthetic_unet_resnet18_vignette_illumination_...,unet,resnet18,none,vignette_illumination,virtual_synthetic_add_back,1,strong_clinical_proxy_alternate,Strong validation Dice with a degradation patt...,4,0.137766,0.849778,0.780482,0.815130,0.069679,0.803803,0.011327,False,True


Saved final selection plan: reports/training/final_model_selection_plan.csv


## 07.08 — Public split manifest summary

Confirm the public train/validation/test split sizes and verify that image and mask paths resolve before building datasets.


In [8]:
# 07.08 — Public split manifest summary
def first_existing_column(frame: pd.DataFrame, candidates: list[str], label: str) -> str:
    for column in candidates:
        if column in frame.columns:
            return column
    raise KeyError(f"Could not find {label} column. Tried: {candidates}")


def resolve_manifest_path(path_value: Any) -> Path:
    path = Path(str(path_value))
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


manifest = pd.read_csv(MANIFEST_PATH)

split_column = first_existing_column(manifest, ["split", "split_name", "partition"], "split")
image_column = first_existing_column(manifest, ["image_path", "image_filepath", "image_file", "image"], "image path")
mask_column = first_existing_column(manifest, ["mask_path", "mask_filepath", "mask_file", "mask"], "mask path")

source_column = None
for candidate in ["source_dataset", "dataset", "source", "dataset_name"]:
    if candidate in manifest.columns:
        source_column = candidate
        break

split_summary = (
    manifest.groupby(split_column)
    .size()
    .rename("n_rows")
    .reset_index()
    .sort_values(split_column)
)

expected_split_counts = {"train": 1911, "val": 725, "test": 722}
actual_split_counts = dict(zip(split_summary[split_column], split_summary["n_rows"]))

for split_name, expected_count in expected_split_counts.items():
    actual_count = int(actual_split_counts.get(split_name, -1))
    if actual_count != expected_count:
        raise ValueError(
            f"Unexpected {split_name!r} split count. Expected {expected_count}, found {actual_count}."
        )

image_exists = manifest[image_column].map(lambda value: resolve_manifest_path(value).exists())
mask_exists = manifest[mask_column].map(lambda value: resolve_manifest_path(value).exists())

path_summary = pd.DataFrame(
    [
        {"path_type": "image", "n_missing": int((~image_exists).sum()), "n_present": int(image_exists.sum())},
        {"path_type": "mask", "n_missing": int((~mask_exists).sum()), "n_present": int(mask_exists.sum())},
    ]
)

if int(path_summary["n_missing"].sum()) > 0:
    missing_examples = manifest.loc[(~image_exists) | (~mask_exists), [image_column, mask_column]].head()
    display(missing_examples)
    raise FileNotFoundError("One or more public manifest image/mask paths do not resolve.")

if source_column is not None:
    source_split_summary = (
        manifest.groupby([source_column, split_column])
        .size()
        .rename("n_rows")
        .reset_index()
        .sort_values([source_column, split_column])
    )
else:
    source_split_summary = pd.DataFrame()

dataset_summary = split_summary.copy()
dataset_summary["manifest_path"] = str(MANIFEST_PATH.relative_to(PROJECT_ROOT))
dataset_summary["image_column"] = image_column
dataset_summary["mask_column"] = mask_column
dataset_summary["source_column"] = source_column if source_column is not None else "not_available"
dataset_summary.to_csv(FINAL_SELECTION_DATASET_SUMMARY_PATH, index=False)

print("Split summary")
display(split_summary)

print("Path summary")
display(path_summary)

if not source_split_summary.empty:
    print("Source-by-split summary")
    display(source_split_summary)

print(f"Saved dataset summary: {FINAL_SELECTION_DATASET_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")


Split summary


,split,n_rows
0,test,722
1,train,1911
2,val,725


Path summary


,path_type,n_missing,n_present
0,image,0,3358
1,mask,0,3358


Source-by-split summary


,source_dataset,split,n_rows
0,G1020,test,153
1,G1020,train,714
2,G1020,val,153
3,ORIGA,test,97
4,ORIGA,train,455
5,ORIGA,val,98
6,PAPILA,test,72
7,PAPILA,train,342
8,PAPILA,val,74
9,REFUGE,test,400


Saved dataset summary: reports/data_audit/final_model_selection_dataset_summary.csv


## 07.09 — Notebook 07 setup handoff summary

Summarize the setup checks completed before dataset construction, training, and held-out public test evaluation.


In [9]:
# 07.09 — Notebook 07 setup handoff summary
setup_handoff = pd.DataFrame(
    [
        {
            "check": "repository_inputs",
            "status": "ready",
            "detail": "Required manifest and prior Notebook 04–06 summary files are present.",
        },
        {
            "check": "candidate_plan",
            "status": "ready",
            "detail": f"Saved {FINAL_SELECTION_PLAN_PATH.relative_to(PROJECT_ROOT)} with {len(FINALIST_CONFIGS)} finalists.",
        },
        {
            "check": "public_manifest",
            "status": "ready",
            "detail": "Train/validation/test split counts match the expected public-data split.",
        },
        {
            "check": "path_resolution",
            "status": "ready",
            "detail": "All public image and mask paths in the combined split manifest resolve locally.",
        },
        {
            "check": "next_step",
            "status": "pending",
            "detail": "Build source-backed train/validation/test datasets and DataLoaders before finalist training.",
        },
    ]
)

display(setup_handoff)

print("Notebook 07 setup cells are ready. Next: dataset construction and DataLoader smoke tests.")


,check,status,detail
0,repository_inputs,ready,Required manifest and prior Notebook 04–06 sum...
1,candidate_plan,ready,Saved reports/training/final_model_selection_p...
2,public_manifest,ready,Train/validation/test split counts match the e...
3,path_resolution,ready,All public image and mask paths in the combine...
4,next_step,pending,Build source-backed train/validation/test data...


Notebook 07 setup cells are ready. Next: dataset construction and DataLoader smoke tests.


## 07.10 — Build source-backed public split datasets

Build the original public train, validation, and test datasets from the combined split manifest. Validation and test remain unaltered throughout Notebook 07.


In [10]:
# 07.10 — Build source-backed public split datasets
public_datasets = make_segmentation_datasets(
    manifest_path=MANIFEST_PATH,
    splits=("train", "val", "test"),
    image_size=IMAGE_SIZE,
    validate_paths=True,
    validate_masks=True,
)

public_dataset_lengths = {
    "train": len(public_datasets.train),
    "val": len(public_datasets.val),
    "test": len(public_datasets.test),
}

expected_public_dataset_lengths = {
    "train": 1911,
    "val": 725,
    "test": 722,
}

if public_dataset_lengths != expected_public_dataset_lengths:
    raise ValueError(
        "Unexpected public dataset lengths. "
        f"Expected {expected_public_dataset_lengths}, found {public_dataset_lengths}."
    )

sample = public_datasets.train[0]
sample_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "row_index": 0,
            "sample_keys": sorted(sample.keys()),
            "image_shape": tuple(sample["image"].shape),
            "image_dtype": str(sample["image"].dtype),
            "image_min": float(sample["image"].min()),
            "image_max": float(sample["image"].max()),
            "mask_shape": tuple(sample["mask"].shape),
            "mask_dtype": str(sample["mask"].dtype),
            "mask_values": sorted(int(value) for value in torch.unique(sample["mask"]).tolist()),
        }
    ]
)

split_dataset_summary = pd.DataFrame(
    [
        {
            "dataset_label": f"public_{split_name}_original",
            "split": split_name,
            "rows": row_count,
            "image_size": IMAGE_SIZE,
            "is_virtual_synthetic": False,
            "strategy_name": "none",
            "synthetic_copy_count": 0,
            "materializes_synthetic_files": False,
            "used_for": "training" if split_name == "train" else "evaluation",
        }
        for split_name, row_count in public_dataset_lengths.items()
    ]
)

display(split_dataset_summary)
display(sample_summary)

print("Public split dataset construction: OK")


,dataset_label,split,rows,image_size,is_virtual_synthetic,strategy_name,synthetic_copy_count,materializes_synthetic_files,used_for
0,public_train_original,train,1911,"(256, 256)",False,none,0,False,training
1,public_val_original,val,725,"(256, 256)",False,none,0,False,evaluation
2,public_test_original,test,722,"(256, 256)",False,none,0,False,evaluation


,split,row_index,sample_keys,image_shape,image_dtype,image_min,image_max,mask_shape,mask_dtype,mask_values
0,train,0,"[dataset_key, file_id, image, image_path, mask...","(3, 256, 256)",torch.float32,0.0,1.0,"(256, 256)",torch.int64,"[0, 1, 2]"


Public split dataset construction: OK


## 07.11 — Build finalist virtual synthetic training datasets

Wrap only the public training split with deterministic virtual synthetic add-back for each finalist recipe. Validation and test datasets remain original.


In [11]:
# 07.11 — Build finalist virtual synthetic training datasets
finalist_train_datasets: dict[str, Any] = {}
virtual_dataset_rows: list[dict[str, Any]] = []

for config in FINALIST_CONFIGS:
    run_name = config["final_run_name"]

    virtual_train_dataset = build_virtual_synthetic_expansion_dataset(
        base_dataset=public_datasets.train,
        strategy_names=config["strategy_name"],
        copy_count=int(config["synthetic_copy_count"]),
        base_seed=SEED,
        include_original=True,
        add_metadata=True,
    )

    finalist_train_datasets[run_name] = virtual_train_dataset

    summary = summarize_virtual_synthetic_expansion(virtual_train_dataset)
    virtual_dataset_rows.append(
        {
            "dataset_label": f"{run_name}_train_virtual",
            "split": "train",
            "rows": len(virtual_train_dataset),
            "image_size": IMAGE_SIZE,
            "is_virtual_synthetic": True,
            "strategy_name": config["strategy_name"],
            "synthetic_copy_count": int(config["synthetic_copy_count"]),
            "materializes_synthetic_files": False,
            "used_for": "training",
            "base_rows": summary["base_rows"],
            "original_rows_exposed": summary["original_rows_exposed"],
            "synthetic_rows_exposed": summary["synthetic_rows_exposed"],
            "total_rows_exposed": summary["total_rows_exposed"],
            "base_seed": summary["base_seed"],
            "include_original": summary["include_original"],
        }
    )

virtual_dataset_summary = pd.DataFrame(virtual_dataset_rows)

final_dataset_summary = pd.concat(
    [split_dataset_summary, virtual_dataset_summary],
    ignore_index=True,
    sort=False,
)

final_dataset_summary.to_csv(FINAL_SELECTION_DATASET_SUMMARY_PATH, index=False)

display(virtual_dataset_summary)
display(final_dataset_summary)

for config in FINALIST_CONFIGS:
    run_name = config["final_run_name"]
    dataset = finalist_train_datasets[run_name]
    expected_rows = len(public_datasets.train) * (1 + int(config["synthetic_copy_count"]))
    if len(dataset) != expected_rows:
        raise ValueError(
            f"Unexpected virtual train length for {run_name}: "
            f"expected {expected_rows}, found {len(dataset)}."
        )

print(f"Saved dataset summary: {FINAL_SELECTION_DATASET_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print("Finalist virtual synthetic training datasets: OK")


,dataset_label,split,rows,image_size,is_virtual_synthetic,strategy_name,synthetic_copy_count,materializes_synthetic_files,used_for,base_rows,original_rows_exposed,synthetic_rows_exposed,total_rows_exposed,base_seed,include_original
0,final_unetplusplus_resnet18_small_affine_virtu...,train,3822,"(256, 256)",True,small_affine,1,False,training,1911,1911,1911,3822,42,True
1,final_unet_resnet18_vignette_virtual_synthetic...,train,3822,"(256, 256)",True,vignette_illumination,1,False,training,1911,1911,1911,3822,42,True


,dataset_label,split,rows,image_size,is_virtual_synthetic,strategy_name,synthetic_copy_count,materializes_synthetic_files,used_for,base_rows,original_rows_exposed,synthetic_rows_exposed,total_rows_exposed,base_seed,include_original
0,public_train_original,train,1911,"(256, 256)",False,none,0,False,training,NaN,NaN,NaN,NaN,NaN,NaN
1,public_val_original,val,725,"(256, 256)",False,none,0,False,evaluation,NaN,NaN,NaN,NaN,NaN,NaN
2,public_test_original,test,722,"(256, 256)",False,none,0,False,evaluation,NaN,NaN,NaN,NaN,NaN,NaN
3,final_unetplusplus_resnet18_small_affine_virtu...,train,3822,"(256, 256)",True,small_affine,1,False,training,1911.0,1911.0,1911.0,3822.0,42.0,True
4,final_unet_resnet18_vignette_virtual_synthetic...,train,3822,"(256, 256)",True,vignette_illumination,1,False,training,1911.0,1911.0,1911.0,3822.0,42.0,True


Saved dataset summary: reports/data_audit/final_model_selection_dataset_summary.csv
Finalist virtual synthetic training datasets: OK


## 07.12 — DataLoader smoke tests

Build DataLoaders for finalist training and original validation/test evaluation. Check tensor shapes, mask values, and virtual-synthetic metadata collation before training.


In [12]:
# 07.12 — DataLoader smoke tests
def build_loader(
    dataset: Any,
    *,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    pin_memory: bool,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
        persistent_workers=num_workers > 0,
    )


val_loader = build_loader(
    public_datasets.val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = build_loader(
    public_datasets.test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

finalist_loaders: dict[str, dict[str, DataLoader]] = {}

for config in FINALIST_CONFIGS:
    run_name = config["final_run_name"]

    finalist_loaders[run_name] = {
        "train": build_loader(
            finalist_train_datasets[run_name],
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        ),
        "val": val_loader,
        "test": test_loader,
    }


def summarize_collated_value(value: Any) -> str:
    if value is None:
        return "not_available"
    if torch.is_tensor(value):
        unique_values = torch.unique(value.detach().cpu()).tolist()
        return str(unique_values[:10])
    if isinstance(value, list):
        unique_values = sorted(set(str(item) for item in value))
        return str(unique_values[:10])
    return str(type(value))


def summarize_batch(batch: Any, *, loader_name: str, expected_batch_size: int) -> dict[str, Any]:
    images, masks = extract_images_and_masks(batch)

    if images.ndim != 4:
        raise ValueError(f"{loader_name} images must have shape (B, C, H, W), got {tuple(images.shape)}.")
    if masks.ndim != 3:
        raise ValueError(f"{loader_name} masks must have shape (B, H, W), got {tuple(masks.shape)}.")
    if images.shape[0] > expected_batch_size:
        raise ValueError(f"{loader_name} batch exceeds expected batch size {expected_batch_size}.")
    if images.shape[1] != 3:
        raise ValueError(f"{loader_name} images should have 3 channels, got {images.shape[1]}.")
    if tuple(images.shape[2:]) != tuple(reversed(IMAGE_SIZE)):
        raise ValueError(
            f"{loader_name} image tensor spatial shape mismatch. "
            f"Expected {tuple(reversed(IMAGE_SIZE))}, got {tuple(images.shape[2:])}."
        )
    if tuple(masks.shape[1:]) != tuple(reversed(IMAGE_SIZE)):
        raise ValueError(
            f"{loader_name} mask tensor spatial shape mismatch. "
            f"Expected {tuple(reversed(IMAGE_SIZE))}, got {tuple(masks.shape[1:])}."
        )

    mask_values = sorted(int(value) for value in torch.unique(masks).tolist())
    if not set(mask_values).issubset({0, 1, 2}):
        raise ValueError(f"{loader_name} masks contain unexpected values: {mask_values}.")

    return {
        "loader_name": loader_name,
        "batch_image_shape": tuple(images.shape),
        "batch_mask_shape": tuple(masks.shape),
        "image_dtype": str(images.dtype),
        "mask_dtype": str(masks.dtype),
        "image_min": float(images.min()),
        "image_max": float(images.max()),
        "mask_values": mask_values,
        "is_synthetic_values": summarize_collated_value(batch.get("is_synthetic") if isinstance(batch, dict) else None),
        "synthetic_strategy_values": summarize_collated_value(batch.get("synthetic_strategy") if isinstance(batch, dict) else None),
    }


loader_rows: list[dict[str, Any]] = []

for config in FINALIST_CONFIGS:
    run_name = config["final_run_name"]
    loader_rows.append(
        summarize_batch(
            next(iter(finalist_loaders[run_name]["train"])),
            loader_name=f"{run_name}:train",
            expected_batch_size=BATCH_SIZE,
        )
    )

loader_rows.append(
    summarize_batch(
        next(iter(val_loader)),
        loader_name="public_original:val",
        expected_batch_size=BATCH_SIZE,
    )
)

loader_rows.append(
    summarize_batch(
        next(iter(test_loader)),
        loader_name="public_original:test",
        expected_batch_size=BATCH_SIZE,
    )
)

loader_smoke_summary = pd.DataFrame(loader_rows)

loader_length_summary = pd.DataFrame(
    [
        {
            "run_name": config["final_run_name"],
            "train_batches": len(finalist_loaders[config["final_run_name"]]["train"]),
            "val_batches": len(val_loader),
            "test_batches": len(test_loader),
            "train_rows": len(finalist_train_datasets[config["final_run_name"]]),
            "val_rows": len(public_datasets.val),
            "test_rows": len(public_datasets.test),
        }
        for config in FINALIST_CONFIGS
    ]
)

display(loader_length_summary)
display(loader_smoke_summary)

print("DataLoader smoke tests: OK")


,run_name,train_batches,val_batches,test_batches,train_rows,val_rows,test_rows
0,final_unetplusplus_resnet18_small_affine_virtu...,478,91,91,3822,725,722
1,final_unet_resnet18_vignette_virtual_synthetic...,478,91,91,3822,725,722


,loader_name,batch_image_shape,batch_mask_shape,image_dtype,mask_dtype,image_min,image_max,mask_values,is_synthetic_values,synthetic_strategy_values
0,final_unetplusplus_resnet18_small_affine_virtu...,"(8, 3, 256, 256)","(8, 256, 256)",torch.float32,torch.int64,0.0,1.0,"[0, 1, 2]","[False, True]","['none', 'small_affine']"
1,final_unet_resnet18_vignette_virtual_synthetic...,"(8, 3, 256, 256)","(8, 256, 256)",torch.float32,torch.int64,0.0,1.0,"[0, 1, 2]","[False, True]","['none', 'vignette_illumination']"
2,public_original:val,"(8, 3, 256, 256)","(8, 256, 256)",torch.float32,torch.int64,0.0,1.0,"[0, 1, 2]",not_available,not_available
3,public_original:test,"(8, 3, 256, 256)","(8, 256, 256)",torch.float32,torch.int64,0.0,1.0,"[0, 1, 2]",not_available,not_available


DataLoader smoke tests: OK


## 07.13 — Model and loss smoke test

Build the primary finalist model and run one forward/loss/metric pass on a small batch. This checks model output shape and loss compatibility before full finalist training.


In [13]:
# 07.13 — Model and loss smoke test
primary_config = FINALIST_CONFIGS[0]
primary_run_name = primary_config["final_run_name"]

smoke_model = build_model(
    model_name=primary_config["model_name"],
    in_channels=3,
    classes=3,
    encoder_name=primary_config["encoder_name"],
    encoder_weights=primary_config["encoder_weights"],
    activation=None,
).to(DEVICE)

smoke_criterion = DiceCELoss()

smoke_batch = next(iter(finalist_loaders[primary_run_name]["train"]))
smoke_images, smoke_masks = extract_images_and_masks(smoke_batch)

smoke_images = smoke_images[:2].to(DEVICE, non_blocking=True).float()
smoke_masks = smoke_masks[:2].to(DEVICE, non_blocking=True).long()

smoke_model.eval()
with torch.no_grad():
    smoke_logits = model_forward(smoke_model, smoke_images)
    smoke_loss = smoke_criterion(smoke_logits, smoke_masks)

if smoke_logits.shape != (smoke_images.shape[0], 3, smoke_images.shape[2], smoke_images.shape[3]):
    raise ValueError(
        "Unexpected smoke-test logits shape. "
        f"Expected {(smoke_images.shape[0], 3, smoke_images.shape[2], smoke_images.shape[3])}, "
        f"found {tuple(smoke_logits.shape)}."
    )

if not torch.isfinite(smoke_loss):
    raise ValueError(f"Smoke-test loss is not finite: {float(smoke_loss.detach().cpu())}")

smoke_metrics = SegMetrics()
smoke_metrics.update(smoke_logits.detach(), smoke_masks.detach())
smoke_metric_summary = smoke_metrics.compute()

smoke_summary = pd.DataFrame(
    [
        {
            "run_name": primary_run_name,
            "model_name": primary_config["model_name"],
            "encoder_name": primary_config["encoder_name"],
            "encoder_weights": "none" if primary_config["encoder_weights"] is None else primary_config["encoder_weights"],
            "device": str(DEVICE),
            "input_shape": tuple(smoke_images.shape),
            "logits_shape": tuple(smoke_logits.shape),
            "loss": float(smoke_loss.detach().cpu()),
            "disc_dice": smoke_metric_summary["disc_dice"],
            "cup_dice": smoke_metric_summary["cup_dice"],
            "cdr_mae": smoke_metric_summary["cdr_mae"],
            "n_images": smoke_metric_summary["n_images"],
        }
    ]
)

display(smoke_summary)

del smoke_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Model and loss smoke test: OK")
print("Notebook 07 is ready for finalist training cells.")


,run_name,model_name,encoder_name,encoder_weights,device,input_shape,logits_shape,loss,disc_dice,cup_dice,cdr_mae,n_images
0,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,none,cuda,"(2, 3, 256, 256)","(2, 3, 256, 256)",1.856288,0.00344,0.000667,0.764151,2.0


Model and loss smoke test: OK
Notebook 07 is ready for finalist training cells.


## 07.14 — Finalist training and held-out test evaluation helpers

Train one finalist recipe at a time, retain the Dice-first best-validation model state in memory, and evaluate that state once on the held-out public test split.


In [14]:
# 07.14 — Finalist training and held-out test evaluation helpers
def mean_foreground_dice(disc_dice: float, cup_dice: float) -> float:
    """Compute the primary Dice-first selection metric."""
    return float((float(disc_dice) + float(cup_dice)) / 2.0)


def add_mean_foreground_dice(frame: pd.DataFrame) -> pd.DataFrame:
    """Add mean foreground Dice to any frame with disc_dice and cup_dice columns."""
    frame = frame.copy()
    frame["mean_foreground_dice"] = (
        frame["disc_dice"].astype(float) + frame["cup_dice"].astype(float)
    ) / 2.0
    return frame


def load_csv_if_exists(path: Path) -> pd.DataFrame:
    """Load a CSV if it exists; otherwise return an empty dataframe."""
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def remove_run_if_present(frame: pd.DataFrame, run_name: str) -> pd.DataFrame:
    """Remove an existing run from a dataframe before overwriting it."""
    if frame.empty or "run_name" not in frame.columns:
        return frame
    return frame.loc[frame["run_name"] != run_name].copy()


def append_run_rows(path: Path, new_rows: pd.DataFrame, run_name: str, overwrite: bool) -> pd.DataFrame:
    """Append run rows to a CSV, optionally replacing prior rows for the same run."""
    existing = load_csv_if_exists(path)

    if overwrite:
        existing = remove_run_if_present(existing, run_name)

    combined = pd.concat([existing, new_rows], ignore_index=True)
    combined.to_csv(path, index=False)
    return combined


def final_config_for_run(run_name: str) -> dict[str, Any]:
    """Return the finalist config for a final run name."""
    matches = [config for config in FINALIST_CONFIGS if config["final_run_name"] == run_name]

    if len(matches) != 1:
        raise ValueError(f"Expected exactly one finalist config for {run_name!r}; found {len(matches)}.")

    return matches[0]


def build_train_loader_for_finalist(run_name: str, *, candidate_priority: int) -> DataLoader:
    """Build a deterministic shuffled train loader for one finalist."""
    train_generator = torch.Generator()
    train_generator.manual_seed(SEED + int(candidate_priority))

    return DataLoader(
        finalist_train_datasets[run_name],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=train_generator,
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
    )


def build_finalist_model(config: dict[str, Any]) -> torch.nn.Module:
    """Build one finalist segmentation model from its config."""
    return build_model(
        model_name=config["model_name"],
        in_channels=3,
        classes=3,
        encoder_name=config["encoder_name"],
        encoder_weights=config["encoder_weights"],
        activation=None,
    )


def epoch_result_rows(
    history: list[Any],
    *,
    config: dict[str, Any],
    run_name: str,
) -> pd.DataFrame:
    """Convert EpochResult history to a run-level dataframe."""
    frame = pd.DataFrame(history_to_dicts(history))
    frame.insert(0, "run_name", run_name)
    frame.insert(1, "model_name", config["model_name"])
    frame.insert(2, "encoder_name", config["encoder_name"])
    frame.insert(3, "strategy_name", config["strategy_name"])
    frame.insert(4, "augmentation_mode", config["augmentation_mode"])
    frame.insert(5, "candidate_role", config["candidate_role"])
    frame = add_mean_foreground_dice(frame)
    return frame


def summarize_best_validation_from_history(history_frame: pd.DataFrame, run_name: str) -> pd.DataFrame:
    """Return the best validation row for one finalist by mean foreground Dice."""
    run_history = history_frame.loc[history_frame["run_name"] == run_name].copy()
    val_history = run_history.loc[run_history["phase"] == "val"].copy()

    if val_history.empty:
        return pd.DataFrame()

    if "mean_foreground_dice" not in val_history.columns:
        val_history = add_mean_foreground_dice(val_history)

    val_history = val_history.sort_values(
        ["mean_foreground_dice", "cup_dice", "disc_dice"],
        ascending=[False, False, False],
    )

    best = val_history.head(1).copy()

    best = best.rename(
        columns={
            "epoch": "best_epoch_by_val_mean_foreground_dice",
            "loss": "best_val_loss",
            "disc_dice": "best_val_disc_dice",
            "cup_dice": "best_val_cup_dice",
            "cdr_mae": "best_val_cdr_mae",
            "mean_foreground_dice": "val_mean_foreground_dice",
            "n_images": "val_images",
            "n_batches": "val_batches",
        }
    )

    return best[
        [
            "run_name",
            "model_name",
            "encoder_name",
            "strategy_name",
            "augmentation_mode",
            "candidate_role",
            "best_epoch_by_val_mean_foreground_dice",
            "best_val_loss",
            "best_val_disc_dice",
            "best_val_cup_dice",
            "val_mean_foreground_dice",
            "best_val_cdr_mae",
            "val_images",
            "val_batches",
        ]
    ]


def summarize_test_result(test_result: Any, *, config: dict[str, Any], run_name: str, best_epoch: int) -> pd.DataFrame:
    """Summarize one held-out public test evaluation result."""
    row = test_result.to_dict()
    return pd.DataFrame(
        [
            {
                "run_name": run_name,
                "model_name": config["model_name"],
                "encoder_name": config["encoder_name"],
                "strategy_name": config["strategy_name"],
                "augmentation_mode": config["augmentation_mode"],
                "candidate_role": config["candidate_role"],
                "best_epoch_by_val_mean_foreground_dice": int(best_epoch),
                "test_loss": float(row["loss"]),
                "test_disc_dice": float(row["disc_dice"]),
                "test_cup_dice": float(row["cup_dice"]),
                "test_mean_foreground_dice": mean_foreground_dice(row["disc_dice"], row["cup_dice"]),
                "test_cdr_mae": float(row["cdr_mae"]),
                "test_images": int(row["n_images"]),
                "test_batches": int(row["n_batches"]),
            }
        ]
    )


def run_finalist_training_and_test(
    run_name: str,
    *,
    overwrite: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Train one finalist, restore best-validation weights, and evaluate the public test split once."""
    existing_test_summary = load_csv_if_exists(FINAL_SELECTION_TEST_SUMMARY_PATH)

    if (
        not overwrite
        and not existing_test_summary.empty
        and "run_name" in existing_test_summary.columns
        and run_name in set(existing_test_summary["run_name"])
    ):
        print(f"Existing held-out test result found for {run_name}. Set overwrite=True to rerun.")
        existing_validation_summary = load_csv_if_exists(FINAL_SELECTION_VALIDATION_SUMMARY_PATH)
        validation_summary = existing_validation_summary.loc[
            existing_validation_summary["run_name"] == run_name
        ].copy()
        test_summary = existing_test_summary.loc[
            existing_test_summary["run_name"] == run_name
        ].copy()
        display(validation_summary)
        display(test_summary)
        return validation_summary, test_summary

    config = final_config_for_run(run_name)
    train_loader = build_train_loader_for_finalist(
        run_name,
        candidate_priority=int(config["candidate_priority"]),
    )

    print(f"Run name: {run_name}")
    print(f"Model: {config['model_name']} / {config['encoder_name']}")
    print(f"Virtual synthetic strategy: {config['strategy_name']}")
    print(f"Training rows exposed: {len(finalist_train_datasets[run_name]):,}")
    print(f"Validation rows: {len(public_datasets.val):,}")
    print(f"Held-out test rows: {len(public_datasets.test):,}")
    print(f"Device: {DEVICE}")

    seed_everything(SEED, deterministic=False)

    model = build_finalist_model(config).to(DEVICE)
    criterion = DiceCELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    history: list[Any] = []
    best_state_dict: dict[str, torch.Tensor] | None = None
    best_validation_score = -math.inf
    best_validation_epoch: int | None = None
    best_validation_tiebreaker: tuple[float, float] = (-math.inf, -math.inf)

    start_time = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        print("=" * 90)
        print(
            f"Running {run_name} | epoch {epoch:02d}/{EPOCHS:02d} | "
            f"image_size={IMAGE_SIZE} | batch_size={BATCH_SIZE}"
        )
        print("=" * 90)

        train_result = run_one_epoch_with_progress(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            epoch=epoch,
            phase="train",
            total_epochs=EPOCHS,
            leave_progress=False,
        )
        history.append(train_result)

        val_result = run_one_epoch_with_progress(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            optimizer=None,
            device=DEVICE,
            epoch=epoch,
            phase="val",
            total_epochs=EPOCHS,
            leave_progress=False,
        )
        history.append(val_result)

        validation_score = mean_foreground_dice(val_result.disc_dice, val_result.cup_dice)
        validation_tiebreaker = (float(val_result.cup_dice), float(val_result.disc_dice))

        if (
            validation_score > best_validation_score
            or (
                math.isclose(validation_score, best_validation_score)
                and validation_tiebreaker > best_validation_tiebreaker
            )
        ):
            best_validation_score = validation_score
            best_validation_tiebreaker = validation_tiebreaker
            best_validation_epoch = epoch
            best_state_dict = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        epoch_minutes = (train_result.elapsed_seconds + val_result.elapsed_seconds) / 60.0

        print(
            f"Epoch {epoch:02d}/{EPOCHS:02d} complete | "
            f"train_loss={train_result.loss:.4f} | "
            f"val_loss={val_result.loss:.4f} | "
            f"val_disc_dice={val_result.disc_dice:.4f} | "
            f"val_cup_dice={val_result.cup_dice:.4f} | "
            f"val_mean_foreground_dice={validation_score:.4f} | "
            f"val_cdr_mae={val_result.cdr_mae:.4f} | "
            f"epoch_time={epoch_minutes:.2f} min"
        )

    if best_state_dict is None or best_validation_epoch is None:
        raise RuntimeError(f"No best validation state was captured for {run_name}.")

    model.load_state_dict(best_state_dict)
    model.to(DEVICE)

    test_result = run_one_epoch_with_progress(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        optimizer=None,
        device=DEVICE,
        epoch=int(best_validation_epoch),
        phase="test",
        total_epochs=EPOCHS,
        leave_progress=False,
    )

    elapsed_minutes = (time.perf_counter() - start_time) / 60.0

    history_frame = epoch_result_rows(history, config=config, run_name=run_name)
    append_run_rows(
        FINAL_SELECTION_HISTORY_PATH,
        history_frame,
        run_name,
        overwrite=overwrite,
    )

    all_history = load_csv_if_exists(FINAL_SELECTION_HISTORY_PATH)
    validation_summary = summarize_best_validation_from_history(all_history, run_name)
    append_run_rows(
        FINAL_SELECTION_VALIDATION_SUMMARY_PATH,
        validation_summary,
        run_name,
        overwrite=overwrite,
    )

    test_summary = summarize_test_result(
        test_result,
        config=config,
        run_name=run_name,
        best_epoch=int(best_validation_epoch),
    )
    append_run_rows(
        FINAL_SELECTION_TEST_SUMMARY_PATH,
        test_summary,
        run_name,
        overwrite=overwrite,
    )

    metadata_frame = pd.DataFrame(
        [
            {
                "run_name": run_name,
                "model_name": config["model_name"],
                "encoder_name": config["encoder_name"],
                "strategy_name": config["strategy_name"],
                "augmentation_mode": config["augmentation_mode"],
                "candidate_role": config["candidate_role"],
                "epochs": EPOCHS,
                "best_epoch_by_val_mean_foreground_dice": int(best_validation_epoch),
                "image_size": str(IMAGE_SIZE),
                "batch_size": BATCH_SIZE,
                "num_workers": NUM_WORKERS,
                "pin_memory": PIN_MEMORY,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "original_train_rows": len(public_datasets.train),
                "virtual_train_rows": len(finalist_train_datasets[run_name]),
                "validation_rows": len(public_datasets.val),
                "test_rows": len(public_datasets.test),
                "synthetic_copy_count": int(config["synthetic_copy_count"]),
                "materializes_synthetic_files": False,
                "total_fit_and_test_minutes": elapsed_minutes,
            }
        ]
    )
    append_run_rows(
        FINAL_SELECTION_METADATA_PATH,
        metadata_frame,
        run_name,
        overwrite=overwrite,
    )

    display(validation_summary)
    display(test_summary)

    del model, optimizer, criterion, best_state_dict
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return validation_summary, test_summary


print("Finalist training/evaluation helpers ready.")
print(f"History path:    {FINAL_SELECTION_HISTORY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Metadata path:   {FINAL_SELECTION_METADATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Validation path: {FINAL_SELECTION_VALIDATION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Test path:       {FINAL_SELECTION_TEST_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")


Finalist training/evaluation helpers ready.
History path:    reports/training/final_model_selection_training_history.csv
Metadata path:   reports/training/final_model_selection_run_metadata.csv
Validation path: reports/training/final_model_selection_validation_summary.csv
Test path:       reports/training/final_model_selection_test_summary.csv


## 07.15 — Train and test primary U-Net++ finalist

Train the U-Net++/ResNet18 small-affine virtual synthetic finalist, restore the best-validation epoch, and evaluate that state once on the held-out public test split.


In [15]:
# 07.15 — Train and test primary U-Net++ finalist
PRIMARY_FINAL_RUN_NAME = "final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch"
OVERWRITE_FINALIST_RUNS = False

primary_validation_summary, primary_test_summary = run_finalist_training_and_test(
    PRIMARY_FINAL_RUN_NAME,
    overwrite=OVERWRITE_FINALIST_RUNS,
)


Run name: final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch
Model: unetplusplus / resnet18
Virtual synthetic strategy: small_affine
Training rows exposed: 3,822
Validation rows: 725
Held-out test rows: 722
Device: cuda
Running final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch | epoch 01/05 | image_size=(256, 256) | batch_size=8


Epoch 01/05 complete | train_loss=1.0048 | val_loss=0.5443 | val_disc_dice=0.6736 | val_cup_dice=0.6466 | val_mean_foreground_dice=0.6601 | val_cdr_mae=0.1627 | epoch_time=3.79 min
Running final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch | epoch 02/05 | image_size=(256, 256) | batch_size=8


Epoch 02/05 complete | train_loss=0.3101 | val_loss=0.2560 | val_disc_dice=0.7200 | val_cup_dice=0.6916 | val_mean_foreground_dice=0.7058 | val_cdr_mae=0.1463 | epoch_time=3.47 min
Running final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch | epoch 03/05 | image_size=(256, 256) | batch_size=8


Epoch 03/05 complete | train_loss=0.1720 | val_loss=0.1546 | val_disc_dice=0.8413 | val_cup_dice=0.7852 | val_mean_foreground_dice=0.8133 | val_cdr_mae=0.0702 | epoch_time=3.44 min
Running final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch | epoch 04/05 | image_size=(256, 256) | batch_size=8


Epoch 04/05 complete | train_loss=0.1380 | val_loss=0.1355 | val_disc_dice=0.8478 | val_cup_dice=0.7981 | val_mean_foreground_dice=0.8229 | val_cdr_mae=0.0624 | epoch_time=3.43 min
Running final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch | epoch 05/05 | image_size=(256, 256) | batch_size=8


Epoch 05/05 complete | train_loss=0.1230 | val_loss=0.1425 | val_disc_dice=0.8461 | val_cup_dice=0.7854 | val_mean_foreground_dice=0.8157 | val_cdr_mae=0.0699 | epoch_time=3.42 min


,run_name,model_name,encoder_name,strategy_name,augmentation_mode,candidate_role,best_epoch_by_val_mean_foreground_dice,best_val_loss,best_val_disc_dice,best_val_cup_dice,val_mean_foreground_dice,best_val_cdr_mae,val_images,val_batches
7,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,virtual_synthetic_add_back,primary_public_test_finalist,4,0.135512,0.847783,0.798071,0.822927,0.06238,725,91


,run_name,model_name,encoder_name,strategy_name,augmentation_mode,candidate_role,best_epoch_by_val_mean_foreground_dice,test_loss,test_disc_dice,test_cup_dice,test_mean_foreground_dice,test_cdr_mae,test_images,test_batches
0,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,virtual_synthetic_add_back,primary_public_test_finalist,4,0.138934,0.83995,0.796002,0.817976,0.063602,722,91


## 07.16 — Train and test alternate U-Net finalist

Train the U-Net/ResNet18 vignette virtual synthetic finalist, restore the best-validation epoch, and evaluate that state once on the held-out public test split.


In [16]:
# 07.16 — Train and test alternate U-Net finalist
ALTERNATE_FINAL_RUN_NAME = "final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch"
OVERWRITE_FINALIST_RUNS = False

alternate_validation_summary, alternate_test_summary = run_finalist_training_and_test(
    ALTERNATE_FINAL_RUN_NAME,
    overwrite=OVERWRITE_FINALIST_RUNS,
)


Run name: final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch
Model: unet / resnet18
Virtual synthetic strategy: vignette_illumination
Training rows exposed: 3,822
Validation rows: 725
Held-out test rows: 722
Device: cuda
Running final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch | epoch 01/05 | image_size=(256, 256) | batch_size=8


Epoch 01/05 complete | train_loss=0.6939 | val_loss=0.4181 | val_disc_dice=0.5560 | val_cup_dice=0.5976 | val_mean_foreground_dice=0.5768 | val_cdr_mae=0.1231 | epoch_time=3.24 min
Running final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch | epoch 02/05 | image_size=(256, 256) | batch_size=8


Epoch 02/05 complete | train_loss=0.2236 | val_loss=0.1718 | val_disc_dice=0.8251 | val_cup_dice=0.7646 | val_mean_foreground_dice=0.7948 | val_cdr_mae=0.0729 | epoch_time=3.15 min
Running final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch | epoch 03/05 | image_size=(256, 256) | batch_size=8


Epoch 03/05 complete | train_loss=0.1560 | val_loss=0.1497 | val_disc_dice=0.8381 | val_cup_dice=0.7732 | val_mean_foreground_dice=0.8056 | val_cdr_mae=0.0737 | epoch_time=3.15 min
Running final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch | epoch 04/05 | image_size=(256, 256) | batch_size=8


Epoch 04/05 complete | train_loss=0.1259 | val_loss=0.1703 | val_disc_dice=0.7839 | val_cup_dice=0.7488 | val_mean_foreground_dice=0.7663 | val_cdr_mae=0.1087 | epoch_time=3.18 min
Running final_unet_resnet18_vignette_virtual_synthetic_256px_5epoch | epoch 05/05 | image_size=(256, 256) | batch_size=8


Epoch 05/05 complete | train_loss=0.1082 | val_loss=0.1345 | val_disc_dice=0.8419 | val_cup_dice=0.7897 | val_mean_foreground_dice=0.8158 | val_cdr_mae=0.0742 | epoch_time=3.19 min


,run_name,model_name,encoder_name,strategy_name,augmentation_mode,candidate_role,best_epoch_by_val_mean_foreground_dice,best_val_loss,best_val_disc_dice,best_val_cup_dice,val_mean_foreground_dice,best_val_cdr_mae,val_images,val_batches
19,final_unet_resnet18_vignette_virtual_synthetic...,unet,resnet18,vignette_illumination,virtual_synthetic_add_back,strong_clinical_proxy_alternate,5,0.134475,0.841867,0.789725,0.815796,0.074157,725,91


,run_name,model_name,encoder_name,strategy_name,augmentation_mode,candidate_role,best_epoch_by_val_mean_foreground_dice,test_loss,test_disc_dice,test_cup_dice,test_mean_foreground_dice,test_cdr_mae,test_images,test_batches
0,final_unet_resnet18_vignette_virtual_synthetic...,unet,resnet18,vignette_illumination,virtual_synthetic_add_back,strong_clinical_proxy_alternate,5,0.136262,0.838077,0.789952,0.814015,0.071707,722,91


## 07.17 — Summarize final public test results

Combine validation and held-out public test results for the finalists. The final public-data model carried forward is selected by held-out test mean foreground Dice.


In [17]:
# 07.17 — Summarize final public test results
validation_summary = load_csv_if_exists(FINAL_SELECTION_VALIDATION_SUMMARY_PATH)
test_summary = load_csv_if_exists(FINAL_SELECTION_TEST_SUMMARY_PATH)
metadata_summary = load_csv_if_exists(FINAL_SELECTION_METADATA_PATH)

expected_final_runs = {config["final_run_name"] for config in FINALIST_CONFIGS}
completed_test_runs = set(test_summary["run_name"]) if not test_summary.empty and "run_name" in test_summary.columns else set()

missing_test_runs = sorted(expected_final_runs - completed_test_runs)
if missing_test_runs:
    raise ValueError(f"Missing held-out test results for final runs: {missing_test_runs}")

final_selection_summary = validation_summary.merge(
    test_summary,
    on=[
        "run_name",
        "model_name",
        "encoder_name",
        "strategy_name",
        "augmentation_mode",
        "candidate_role",
        "best_epoch_by_val_mean_foreground_dice",
    ],
    how="inner",
)

if not metadata_summary.empty:
    final_selection_summary = final_selection_summary.merge(
        metadata_summary[
            [
                "run_name",
                "original_train_rows",
                "virtual_train_rows",
                "validation_rows",
                "test_rows",
                "synthetic_copy_count",
                "materializes_synthetic_files",
                "total_fit_and_test_minutes",
            ]
        ],
        on="run_name",
        how="left",
    )

final_selection_summary = final_selection_summary.merge(
    final_selection_plan[
        [
            "final_run_name",
            "candidate_priority",
            "selection_rationale",
            "validation_mean_foreground_dice",
            "online_reference_mean_foreground_dice",
            "delta_vs_online_reference",
        ]
    ].rename(columns={"final_run_name": "run_name"}),
    on="run_name",
    how="left",
)

final_selection_summary = final_selection_summary.sort_values(
    ["test_mean_foreground_dice", "test_cup_dice", "test_disc_dice"],
    ascending=[False, False, False],
).reset_index(drop=True)

final_selection_summary["public_test_rank"] = np.arange(1, len(final_selection_summary) + 1)

best_test_dice = float(final_selection_summary.loc[0, "test_mean_foreground_dice"])
if len(final_selection_summary) > 1:
    second_test_dice = float(final_selection_summary.loc[1, "test_mean_foreground_dice"])
    test_margin_over_next = best_test_dice - second_test_dice
else:
    test_margin_over_next = float("nan")

final_selection_summary["selected_for_notebook_08_clinical_generalization"] = (
    final_selection_summary["public_test_rank"] == 1
)
final_selection_summary["test_margin_over_next_best"] = test_margin_over_next
final_selection_summary["meaningful_test_margin_over_next"] = (
    test_margin_over_next >= PRIMARY_DICE_IMPROVEMENT_THRESHOLD
    if not math.isnan(test_margin_over_next)
    else False
)

final_selection_summary.to_csv(FINAL_SELECTION_SUMMARY_PATH, index=False)

display_columns = [
    "public_test_rank",
    "run_name",
    "model_name",
    "encoder_name",
    "strategy_name",
    "best_epoch_by_val_mean_foreground_dice",
    "val_mean_foreground_dice",
    "test_mean_foreground_dice",
    "test_disc_dice",
    "test_cup_dice",
    "test_cdr_mae",
    "selected_for_notebook_08_clinical_generalization",
    "test_margin_over_next_best",
    "meaningful_test_margin_over_next",
    "total_fit_and_test_minutes",
]

display(final_selection_summary[display_columns])

selected_row = final_selection_summary.loc[0].to_dict()

print("Final public-data model selected for Notebook 08 clinical generalization:")
print(f"  run_name: {selected_row['run_name']}")
print(f"  model: {selected_row['model_name']} / {selected_row['encoder_name']}")
print(f"  strategy: {selected_row['strategy_name']}")
print(f"  best validation epoch: {int(selected_row['best_epoch_by_val_mean_foreground_dice'])}")
print(f"  held-out test mean foreground Dice: {selected_row['test_mean_foreground_dice']:.4f}")
print(f"  held-out test disc Dice: {selected_row['test_disc_dice']:.4f}")
print(f"  held-out test cup Dice: {selected_row['test_cup_dice']:.4f}")
print(f"  held-out test CDR MAE: {selected_row['test_cdr_mae']:.4f}")
print(f"Saved final summary: {FINAL_SELECTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")


,public_test_rank,run_name,model_name,encoder_name,strategy_name,best_epoch_by_val_mean_foreground_dice,val_mean_foreground_dice,test_mean_foreground_dice,test_disc_dice,test_cup_dice,test_cdr_mae,selected_for_notebook_08_clinical_generalization,test_margin_over_next_best,meaningful_test_margin_over_next,total_fit_and_test_minutes
0,1,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,4,0.822927,0.817976,0.839950,0.796002,0.063602,True,0.003962,False,18.088981
1,2,final_unet_resnet18_vignette_virtual_synthetic...,unet,resnet18,vignette_illumination,5,0.815796,0.814015,0.838077,0.789952,0.071707,False,0.003962,False,16.431262


Final public-data model selected for Notebook 08 clinical generalization:
  run_name: final_unetplusplus_resnet18_small_affine_virtual_synthetic_256px_5epoch
  model: unetplusplus / resnet18
  strategy: small_affine
  best validation epoch: 4
  held-out test mean foreground Dice: 0.8180
  held-out test disc Dice: 0.8400
  held-out test cup Dice: 0.7960
  held-out test CDR MAE: 0.0636
Saved final summary: reports/training/final_model_selection_summary.csv


## 07.18 — Final Notebook 07 handoff summary

Record the Notebook 07 outcome and the next step for clinical generalization.


In [18]:
# 07.18 — Final Notebook 07 handoff summary
selected = final_selection_summary.loc[0].to_dict()

notebook_07_handoff = pd.DataFrame(
    [
        {
            "item": "primary_metric",
            "status": "complete",
            "detail": "Mean foreground Dice, computed as the average of optic disc Dice and optic cup Dice.",
        },
        {
            "item": "held_out_test_policy",
            "status": "complete",
            "detail": "Each finalist was evaluated once on the original public test split after restoring its best-validation epoch.",
        },
        {
            "item": "selected_public_model",
            "status": "complete",
            "detail": (
                f"{selected['run_name']} "
                f"({selected['model_name']} / {selected['encoder_name']}, "
                f"strategy={selected['strategy_name']})"
            ),
        },
        {
            "item": "selected_public_test_performance",
            "status": "complete",
            "detail": (
                f"test_mean_foreground_dice={selected['test_mean_foreground_dice']:.4f}; "
                f"test_disc_dice={selected['test_disc_dice']:.4f}; "
                f"test_cup_dice={selected['test_cup_dice']:.4f}; "
                f"test_cdr_mae={selected['test_cdr_mae']:.4f}"
            ),
        },
        {
            "item": "saved_outputs",
            "status": "complete",
            "detail": (
                "Saved final model selection plan, dataset summary, training history, "
                "run metadata, validation summary, test summary, and final selection summary."
            ),
        },
        {
            "item": "next_notebook",
            "status": "ready",
            "detail": (
                "Notebook 08 should use the selected public-data model recipe as the reference "
                "configuration for clinical/head-mounted ophthalmoscopy generalization checks."
            ),
        },
    ]
)

display(notebook_07_handoff)

print("Notebook 07 complete. Next: Notebook 08 clinical data generalization.")


,item,status,detail
0,primary_metric,complete,"Mean foreground Dice, computed as the average ..."
1,held_out_test_policy,complete,Each finalist was evaluated once on the origin...
2,selected_public_model,complete,final_unetplusplus_resnet18_small_affine_virtu...
3,selected_public_test_performance,complete,test_mean_foreground_dice=0.8180; test_disc_di...
4,saved_outputs,complete,"Saved final model selection plan, dataset summ..."
5,next_notebook,ready,Notebook 08 should use the selected public-dat...


Notebook 07 complete. Next: Notebook 08 clinical data generalization.
